# 🎸 music2chart — Gerador de Charts para Clone Hero com GPU T4 (Google Colab)

Este notebook permite executar o pipeline completo do **music2chart** acelerado por hardware com a **GPU Nvidia Tesla T4** no Google Colab.

---

### ⚠️ Passo Obrigatório: Ativar a GPU T4 no Colab
Antes de executar as células abaixo, certifique-se de que a GPU está ativa:
1. No menu superior, vá em: **Ambiente de execução** (Runtime) ➔ **Alterar tipo de ambiente de execução** (Change runtime type).
2. Em **Acelerador de hardware** (Hardware accelerator), selecione **GPU T4**.
3. Clique em **Salvar**.


### 1. Verificar GPU T4 e CUDA


In [ ]:
!nvidia-smi

import torch

print(f"Versão do PyTorch: {torch.__version__}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Detectada: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Total: {vram_gb:.2f} GB")
else:
    print(
        "⚠️ ATENÇÃO: Nenhuma GPU detectada! Ative a GPU T4 no menu: Runtime > Change runtime type > T4 GPU"
    )

### 2. Configurar o Projeto e Instalar Dependências

Se você fez o upload da pasta do projeto para o Colab (ou compactada como `music2chart.zip`), esta célula descompacta e instala todos os pacotes necessários com aceleração CUDA.


In [ ]:
import os
from pathlib import Path

# Se subiu o projeto como music2chart.zip, descomprime automaticamente:
if os.path.exists("/content/music2chart.zip"):
    !unzip -q -o /content/music2chart.zip -d /content/

# Navega para o diretório raiz do music2chart
if os.path.exists("/content/music2chart"):
    %cd /content/music2chart
else:
    print(f"Diretório atual: {os.getcwd()}")

# 1. Dependências do sistema (FFmpeg e libsndfile para processamento de áudio)
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1

# 2. Instalar setuptools compatível (<72) para resampy/pkg_resources
!pip install --quiet "setuptools<72.0.0"

# 3. Instalar o projeto e suas dependências em modo editável
!pip install --quiet -e .

# 4. Utilitários adicionais
!pip install --quiet pyngrok tqdm

print("✅ Ambiente configurado com sucesso para GPU T4!")

---
## Opção A: Gerar Chart Direto via Python (Upload de Música)

Envie uma música (`.mp3`, `.wav`, `.ogg`, `.flac`) do seu computador. O pipeline executará:
1. **Separação de Stems** com Demucs v4 na GPU.
2. **Transcrição Polifônica** com Spotify Basic Pitch na GPU.
3. **Mapeamento de 5 Frets** e acordes ergonômicos no grid musical de 192 PPQ.
4. **Download Automático** do pacote `.sng` e do `.zip` para o seu computador.


In [ ]:
import os
import zipfile

import torch
from google.colab import files

from config.settings import Settings
from src.pipeline import Music2ChartPipeline

print("Selecione o arquivo de áudio do seu computador (.mp3, .wav, .flac, .ogg):")
uploaded = files.upload()

if uploaded:
    audio_filename = list(uploaded.keys())[0]
    audio_path = Path(audio_filename).resolve()
    print(f"\nArquivo recebido: {audio_filename}")

    # Configuração do Pipeline com GPU
    device_to_use = "cuda" if torch.cuda.is_available() else "cpu"
    settings = Settings(
        DEVICE=device_to_use,
        OUTPUT_DIR=Path("/content/output"),
        STEMS_DIR=Path("/content/stems"),
        UPLOAD_DIR=Path("/content/uploads"),
    )
    settings.ensure_directories()

    pipeline = Music2ChartPipeline(settings=settings)

    # Derivar nome da música e artista a partir do nome do arquivo
    base_name = audio_path.stem
    if " - " in base_name:
        artist, song_name = base_name.split(" - ", 1)
        artist = artist.strip()
        song_name = song_name.strip()
    else:
        artist = "Unknown Artist"
        song_name = base_name

    print(
        f"\n🎸 Gerando chart para: '{song_name}' por '{artist}' (Dispositivo: {device_to_use.upper()})...\n"
    )

    def progress_bar(progress: float, stage: str):
        filled = int(30 * progress)
        bar = "█" * filled + "░" * (30 - filled)
        percent = int(progress * 100)
        print(f"\r[{bar}] {percent:>3}% - {stage:<45}", end="", flush=True)

    results = pipeline.process_song(
        audio_path=audio_path,
        song_name=song_name,
        artist=artist,
        output_format="both",
        progress_callback=progress_bar,
    )

    print("\n\n🎉 Chart gerado com sucesso!")
    print(f"Arquivo .chart: {results.get('chart')}")
    print(f"Pacote .sng: {results.get('sng')}")

    # 1. Download do .sng (se gerado)
    sng_path = results.get("sng")
    if sng_path and sng_path.exists():
        print(f"\nBaixando pacote binário Clone Hero: {sng_path.name}...")
        files.download(str(sng_path))

    # 2. Compacta e baixa o pacote ZIP (notes.chart + song.ini + song.ogg)
    pkg_dir = results.get("package_dir")
    if pkg_dir and pkg_dir.exists():
        zip_output = Path(f"/content/{artist} - {song_name}.zip")
        with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zipf:
            for file_path in pkg_dir.rglob("*"):
                if file_path.is_file() and not file_path.name.endswith(".zip"):
                    zipf.write(file_path, arcname=file_path.relative_to(pkg_dir))
        print(f"Baixando pacote completo .zip: {zip_output.name}...")
        files.download(str(zip_output))
else:
    print("Nenhum arquivo enviado.")

---
## Opção B: Iniciar a Interface Web com Barra de Progresso no Colab

Esta opção inicia o servidor FastAPI do **music2chart** e cria um túnel HTTPS público e seguro (via Cloudflare Tunnel).
Você pode abrir a página no seu navegador, arrastar arquivos e acompanhar a **barra de progresso em tempo real** enquanto o Colab com a GPU T4 processa o áudio!


In [ ]:
import os
import re
import subprocess
import time

# 1. Baixar o binário do cloudflared (sem necessidade de cadastro ou tokens)
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Instalando Cloudflare Tunnel...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# 2. Iniciar o servidor FastAPI localmente na porta 8000
print("Iniciando API music2chart com GPU T4...")
server_proc = subprocess.Popen(
    ["uvicorn", "src.api.app:app", "--host", "127.0.0.1", "--port", "8000"]
)

time.sleep(3)

# 3. Iniciar o túnel público apontando para a porta 8000
print("Criando link público seguro...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n" + "=" * 65)
    print("🚀 INTERFACE WEB ONLINE COM GPU T4:")
    print(f"👉 {public_url}")
    print("=" * 65 + "\n")
    print("Abra o link acima no seu navegador para:")
    print("  - Fazer upload de MP3, WAV, FLAC, OGG")
    print("  - Ver a barra de progresso em tempo real com todas as etapas")
    print("  - Baixar os arquivos .sng e .chart diretamente!")
else:
    print("⚠️ Não foi possível capturar o link automaticamente.")
    print("Execute a Opção A para gerar charts diretamente via código.")

---
## Opção C: Processamento em Lote com Google Drive

Conecte seu Google Drive para processar uma pasta com dezenas de músicas em lote e salvar os charts prontos diretamente no Drive.


In [ ]:
from pathlib import Path

import torch
from google.colab import drive

from config.settings import Settings
from src.pipeline import Music2ChartPipeline

# 1. Montar Google Drive
drive.mount("/content/drive")

# 2. Definir pastas no Drive
INPUT_DIR = Path("/content/drive/MyDrive/music2chart/musicas")
OUTPUT_DIR = Path("/content/drive/MyDrive/music2chart/charts_gerados")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pasta de entrada: {INPUT_DIR}")
print(f"Pasta de saída: {OUTPUT_DIR}")

# 3. Encontrar músicas
supported_exts = {".mp3", ".wav", ".flac", ".ogg"}
audio_files = [f for f in INPUT_DIR.iterdir() if f.suffix.lower() in supported_exts]
print(f"Total de músicas encontradas: {len(audio_files)}")

if audio_files:
    settings = Settings(
        DEVICE="cuda" if torch.cuda.is_available() else "cpu",
        OUTPUT_DIR=OUTPUT_DIR,
        STEMS_DIR=Path("/content/stems"),
        UPLOAD_DIR=Path("/content/uploads"),
    )
    settings.ensure_directories()
    pipeline = Music2ChartPipeline(settings=settings)

    for i, audio_file in enumerate(audio_files, 1):
        print(f"\n[{i}/{len(audio_files)}] Processando: {audio_file.name}...")
        base = audio_file.stem
        artist, song = base.split(" - ", 1) if " - " in base else ("Unknown", base)

        try:
            results = pipeline.process_song(
                audio_path=audio_file,
                song_name=song.strip(),
                artist=artist.strip(),
                output_format="both",
                output_dir=OUTPUT_DIR / f"{artist.strip()} - {song.strip()}",
            )
            print(f"✅ Concluído: {results.get('sng') or results.get('chart')}")
        except Exception as exc:
            print(f"❌ Erro ao processar {audio_file.name}: {exc}")

    print("\n🎉 Processamento em lote finalizado!")
else:
    print(
        f"Nenhum arquivo de áudio encontrado em {INPUT_DIR}. Adicione músicas lá e execute novamente!"
    )